In [54]:
import pandas as pd
import numpy as np
import re

In [55]:
# Importing the dataset and creating a copy
df = pd.read_csv("glassdoor_jobs.csv")
df_copy = df.copy()

df

,Unnamed: 0,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
0,0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),-1
1,1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1
2,2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1
3,3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa..."
4,4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,951,Senior Data Engineer,$72K-$133K (Glassdoor est.),THE CHALLENGE\nEventbrite has a world-class da...,4.4,Eventbrite\n4.4,"Nashville, TN","San Francisco, CA",1001 to 5000 employees,2006,Company - Public,Internet,Information Technology,$100 to $500 million (USD),"See Tickets, TicketWeb, Vendini"
952,952,"Project Scientist - Auton Lab, Robotics Institute",$56K-$91K (Glassdoor est.),The Auton Lab at Carnegie Mellon University is...,2.6,Software Engineering Institute\n2.6,"Pittsburgh, PA","Pittsburgh, PA",501 to 1000 employees,1984,College / University,Colleges & Universities,Education,Unknown / Non-Applicable,-1
953,953,Data Science Manager,$95K-$160K (Glassdoor est.),Data Science ManagerResponsibilities:\n\nOvers...,3.2,"Numeric, LLC\n3.2","Allentown, PA","Chadds Ford, PA",1 to 50 employees,-1,Company - Private,Staffing & Outsourcing,Business Services,$5 to $10 million (USD),-1
954,954,Data Engineer,-1,Loading...\n\nTitle: Data Engineer\n\nLocation...,4.8,IGNW\n4.8,"Austin, TX","Portland, OR",201 to 500 employees,2015,Company - Private,IT Services,Information Technology,$25 to $50 million (USD),Slalom


In [56]:
df_type = df.dtypes
df = df.drop("Unnamed: 0", axis=1)

In [57]:
# The column Job Title converting all jobs to Data Science
pattern_map = {
    r".*Scie.*": "Data Science",
    r".*SCIE.*": "Data Science",
    r".*Data.*": "Data Science",
    r".*Anal.*": "Data Science",
    r".*ANAL.*": "Data Science",
    r".*Mach.*": "Data Science",
}
df["Job Title"] = df["Job Title"].replace(pattern_map, regex=True)

# Unique value of a column
df_unique_jt = df["Job Title"].unique()

In [58]:
# ── Step 1: Flag hourly rows BEFORE cleaning (text still intact)
df["hourly"] = df["Salary Estimate"].apply(lambda x: 1 if "per hour" in x.lower() else 0)

# ── Step 2: Strip everything down to bare number ranges
df["Salary Estimate"] = df["Salary Estimate"].apply(
    lambda x: x.split("(")[0]
    .replace("$", "")
    .replace("Employer Provided Salary:", "")
    .replace("K", "")
    .replace("Per Hour", "")  # ← also remove "Per Hour" text
    .replace("per hour", "")
    .strip()
)

# ── Step 3: Build the hourly mask from the FLAG column (not the text)
mask_hourly = df["hourly"] == 1

# ── Step 4: Extract min/max digits from hourly rows and convert to yearly
if mask_hourly.any():
    hourly_data = (
        df.loc[mask_hourly, "Salary Estimate"]
        .str.extractall(r"(\d+)")
        .unstack(level="match")  # ← explicitly unstack the 'match' level
    )

    # Access the MultiIndex columns correctly
    min_h = pd.to_numeric(hourly_data[(0, 0)], errors="coerce")
    max_h = pd.to_numeric(hourly_data[(0, 1)], errors="coerce")

    # hourly × 2000 hrs/year ÷ 1000 (to match K units) = × 2
    avg_yearly = round(((min_h + max_h) / 2) * 2, 2)

    df.loc[mask_hourly, "Salary Estimate"] = avg_yearly.astype(str)

# ── Step 5: Split ranges like "138-224" into min/max, then average
temp_split = (
    df["Salary Estimate"]
    .str.split("-", expand=True)
    .apply(pd.to_numeric, errors="coerce")  # ← coerce instead of hard astype(float)
)

df["avg_salary"] = temp_split.replace(1, np.nan).mean(axis=1)

# ── Step 6: Replace sentinel 1 values with NaN, then fill with column mean
df.loc[df["avg_salary"] == 1.0, "avg_salary"] = np.nan

overall_mean = round(df["avg_salary"].mean(), 2)
df["avg_salary"] = df["avg_salary"].fillna(overall_mean)

# Copy avg_salary values into Salary Estimate
df["Salary Estimate"] = df["avg_salary"]

# Drop avg_salary and hourly columns
df.drop(columns=["avg_salary", "hourly"], inplace=True)

In [59]:
# Remove the rating from the company name"
df["Company Name"] = df.apply(
    lambda x: x["Company Name"] if x["Rating"] < 0 else x["Company Name"][:-3], axis=1
)

In [60]:
# Location: extract state abbreviation
df["Location"] = df["Location"].apply(lambda x: x.split(",")[1] if "," in x else x)

# Replace long state names with abbreviations (fixed spacing in values)
pattern_map = {
    r".*Los Angeles.*": "LA",
    r".*New Jersey.*": "NJ",
    r".*Virginia.*": "VA",
    r".*Maryland.*": "MD",
    r".*Michigan.*": "MI",
    r".*United States.*": "IL",
    r".*Oregon.*": "OR",
}
df["Location"] = df["Location"].replace(pattern_map, regex=True)

# Strip all whitespace from every value to catch any remaining issues
df["Location"] = df["Location"].str.strip()

df_unique_location = df["Location"].unique()
df_counts = df["Location"].value_counts()

In [61]:
# Headquarters, I'll take just the state
df["Headquarters"] = (
    df["Headquarters"].apply(lambda x: x.split(",")[1] if "," in x else x).str.strip()
)
df_unique_Headquarters = df["Headquarters"].unique()

In [62]:
# Map size ranges to their midpoint values
size_order = {
    "1 to 50 employees": 1,
    "51 to 200 employees": 2,
    "201 to 500 employees": 3,
    "501 to 1000 employees": 4,
    "1001 to 5000 employees": 5,
    "5001 to 10000 employees": 6,
    "10000+ employees": 7,
}

df["Size"] = df["Size"].map(size_order)
df_unique_Size = df["Size"].unique()

In [63]:
# Founded year, we will subtract from the currecnt yeat he founded year
df["Founded"] = df["Founded"].apply(lambda x: x if x < 1 else 2026 - x)

df["Founded"] = df["Founded"].apply(lambda x: 0 if str(x).strip() == "-1" else x)

In [64]:
# Column that contains the job descriptions
description_col = "Job Description"  # change to your column name

# Keywords to search for
keywords = ("SQL", "R", "Python", "Excel", "Tableau", "SAS", "spark", "aws")

# Normalize: lowercase + collapse all whitespace into a single string per job
df["normalized"] = df[description_col].str.lower().str.replace(r"\s+", " ", regex=True)

# Count occurrences of each keyword using word boundary patterns
for kw in keywords:
    pattern = rf"\b{re.escape(kw.lower())}\b"
    df[kw] = df["normalized"].str.count(pattern)

# Total mentions per keyword across all job descriptions
totals = df[list(keywords)].sum()

# Print per-job results
print(df[[description_col] + list(keywords)])

print("\n--- Total mentions per keyword ---")
for kw, count in totals.items():
    jobs_with_kw = (df[kw] > 0).sum()
    print(f"{kw:15} total mentions: {count:4}  |  jobs containing it: {jobs_with_kw} / {len(df)}")

                                       Job Description  SQL  R  Python  Excel  \
0    Data Scientist\nLocation: Albuquerque, NM\nEdu...    0  0       1      1   
1    What You Will Do:\n\nI. General Summary\n\nThe...    0  1       2      0   
2    KnowBe4, Inc. is a high growth information sec...    1  1       1      1   
3    *Organization and Job ID**\nJob ID: 310709\n\n...    0  0       1      0   
4    Data Scientist\nAffinity Solutions / Marketing...    1  1       1      0   
..                                                 ...  ... ..     ...    ...   
951  THE CHALLENGE\nEventbrite has a world-class da...    1  0       1      0   
952  The Auton Lab at Carnegie Mellon University is...    0  0       0      0   
953  Data Science ManagerResponsibilities:\n\nOvers...    0  1       0      0   
954  Loading...\n\nTitle: Data Engineer\n\nLocation...    3  0       1      0   
955  Returning Candidate? Log back in to the Career...    0  0       1      0   

     Tableau  SAS  spark  a

In [65]:
# Revenue: I made a scale for the revenue, there are some missing values.
revenue_order = {
    "Less than $1 million (USD)": 1,
    "$1 to $5 million (USD)": 2,
    "$5 to $10 million (USD)": 3,
    "$10 to $25 million (USD)": 4,
    "$25 to $50 million (USD)": 5,
    "$50 to $100 million (USD)": 6,
    "$100 to $500 million (USD)": 7,
    "$500 million to $1 billion (USD)": 8,
    "$1 to $2 billion (USD)": 9,
    "$2 to $5 billion (USD)": 10,
    "$5 to $10 billion (USD)": 11,
    "$10+ billion (USD)": 12,
}

df["Revenue"] = df["Revenue"].map(revenue_order)

# From -1 put 0
df["Revenue"] = df["Revenue"].apply(lambda x: 0 if str(x).strip() == None else x)
df_unique_Revenue = df["Revenue"].unique()

In [66]:
# Competirors: 1 if there are mentioned and 0 if are not mentioned
df["Competitors"] = df["Competitors"].apply(lambda x: 0 if str(x).strip() == "-1" else 1)

In [67]:
# For "Headquarters", "Size", "Founded","Type of ownership","Industry","Sector","Revenue" change from -1 to 0
cols = ["Headquarters", "Size", "Founded", "Type of ownership", "Industry", "Sector", "Revenue"]
df[cols] = df[cols].replace({"Missing value".strip(): -1, "Unknown".strip(): -1, 0: -1}).fillna(-1)

In [68]:
# Shows the count of missing values for every column
df = df.drop("normalized", axis=1)
print(df.isnull().sum())

Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         0
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
SQL                  0
R                    0
Python               0
Excel                0
Tableau              0
SAS                  0
spark                0
aws                  0
dtype: int64


In [69]:
df_uniq_Type_of_ownership = df["Type of ownership"].unique()

In [70]:
df.to_csv("cleaned_data.csv", index=False)
pd.read_csv("cleaned_data.csv")

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,...,Revenue,Competitors,SQL,R,Python,Excel,Tableau,SAS,spark,aws
0,Data Science,72.00,"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n,NM,CA,4.0,53,Company - Private,...,6.0,0,0,0,1,1,1,1,0,0
1,Data Science,87.50,What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n,MD,MD,7.0,42,Other Organization,...,10.0,0,0,1,2,0,0,0,0,0
2,Data Science,85.00,"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n,FL,FL,4.0,16,Company - Private,...,7.0,0,1,1,1,1,0,1,1,0
3,Data Science,76.50,*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n,WA,WA,5.0,61,Government,...,8.0,1,0,0,1,0,0,0,0,0
4,Data Science,114.50,Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n,NY,NY,2.0,28,Company - Private,...,-1.0,1,1,1,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,Data Science,102.50,THE CHALLENGE\nEventbrite has a world-class da...,4.4,Eventbrite\n,TN,CA,5.0,20,Company - Public,...,7.0,1,1,0,1,0,0,0,1,1
952,Data Science,73.50,The Auton Lab at Carnegie Mellon University is...,2.6,Software Engineering Institute\n,PA,PA,4.0,42,College / University,...,-1.0,0,0,0,0,0,0,0,0,0
953,Data Science,127.50,Data Science ManagerResponsibilities:\n\nOvers...,3.2,"Numeric, LLC\n",PA,PA,1.0,-1,Company - Private,...,3.0,0,0,1,0,0,0,0,0,0
954,Data Science,101.43,Loading...\n\nTitle: Data Engineer\n\nLocation...,4.8,IGNW\n,TX,OR,3.0,11,Company - Private,...,5.0,1,3,0,1,0,0,0,0,0
